# 古文を現代語に訳す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/26_machine_translation/classical_to_modern.ipynb)

NLP クックブック 26-16 ／ 機械翻訳

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

古文を現代語に訳したい。LLM に訳させ、訳の例を示したり、誤訳しやすい語の語釈を与えたりすると訳が良くなるかも試す。

In [1]:
!pip install -q sacrebleu

In [2]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる（大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」を参照）。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

@torch.no_grad()
def chat(prompt, max_new_tokens=160):
    ids = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    out = llm.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip().split("\n")[0]

# （古文, 参考にする現代語訳）の組
data = [
    ("今は昔、竹取の翁といふ者ありけり。", "今となっては昔のことだが、竹取の翁という者がいた。"),
    ("野山にまじりて竹を取りつつ、よろづのことに使ひけり。", "野山に分け入って竹を取っては、いろいろなことに使っていた。"),
    ("春はあけぼの。やうやう白くなりゆく山ぎは、少し明かりて、紫だちたる雲の細くたなびきたる。", "春は明け方がよい。だんだん白くなっていく山ぎわが少し明るくなって、紫がかった雲が細くたなびいているのがよい。"),
    ("つれづれなるままに、日暮らし、硯に向かひて、心にうつりゆくよしなしごとを、そこはかとなく書きつくれば、あやしうこそものぐるほしけれ。",
     "することもなく退屈なのにまかせて、一日中硯に向かって、心に浮かんでは消えていくとりとめのないことを、何となく書きつけていると、妙に気持ちが高ぶってくる。"),
    ("行く河の流れは絶えずして、しかももとの水にあらず。", "流れていく川の流れは絶えることがなく、それでいて、もとの水ではない。"),
    ("男もすなる日記といふものを、女もしてみむとてするなり。", "男も書くという日記というものを、女の私も書いてみようと思って書くのである。"),
]

PLAIN = "次の古文を、意味を変えずに現代語に訳してください。訳文だけを 1 行で出力してください。\n\n古文: {}"
FEWSHOT = ("古文を現代語に訳します。訳文だけを 1 行で出力してください。\n\n古文: いづれの御時にか、女御、更衣あまたさぶらひたまひけるなかに\n現代語訳: どの帝の御代であったか、女御や更衣が大勢お仕えしていらっしゃった中に\n\n"
           "古文: 月日は百代の過客にして、行きかふ年もまた旅人なり。\n現代語訳: 月日は永遠の旅人であり、過ぎ去ってはやって来る年もまた旅人である。\n\n古文: {}\n現代語訳:")
for src, ref in data:                          # 6 文 × 2 通りの訳（T4 ならモデルの読み込みを含めて 1〜2 分）
    print(f"{src}\n   訳 (指示のみ): {chat(PLAIN.format(src))}\n   訳 (例を示す): {chat(FEWSHOT.format(src))}\n   参考: {ref}")

今は昔、竹取の翁といふ者ありけり。
   訳 (指示のみ): 今は昔、竹取の翁という者がいた。
   訳 (例を示す): 今は昔、竹取の翁という者がいた。
   参考: 今となっては昔のことだが、竹取の翁という者がいた。
野山にまじりて竹を取りつつ、よろづのことに使ひけり。
   訳 (指示のみ): 「野や山を巡って竹を採取し、様々な用途に使用した」
   訳 (例を示す): 「野山に入り竹を取りながら、様々なことに使用した」
   参考: 野山に分け入って竹を取っては、いろいろなことに使っていた。
春はあけぼの。やうやう白くなりゆく山ぎは、少し明かりて、紫だちたる雲の細くたなびきたる。
   訳 (指示のみ): 春はあけぼの。徐々に白んでいく山の辺が少し明るくなり、紫色がかった雲が細くたなびいている。
   訳 (例を示す): 春はあけぼの。空が白み始め、山々の境界線がゆっくりと白くなっていく中、少し明るくなった空に、紫色がかった雲が細く棚引いている。
   参考: 春は明け方がよい。だんだん白くなっていく山ぎわが少し明るくなって、紫がかった雲が細くたなびいているのがよい。
つれづれなるままに、日暮らし、硯に向かひて、心にうつりゆくよしなしごとを、そこはかとなく書きつくれば、あやしうこそものぐるほしけれ。
   訳 (指示のみ): 「手持ち無沙汰なので一日中、硯に向かって、心に浮かぶとりとめもない事柄を漫然と書きつけていると、奇妙なほどに熱中してしまう。」
   訳 (例を示す): 「手持ち無沙汰なまま一日中、硯に向かって、心に浮かぶとりとめもない事柄を漫然と書きつけていると、奇妙なほどに熱中してしまう。」
   参考: することもなく退屈なのにまかせて、一日中硯に向かって、心に浮かんでは消えていくとりとめのないことを、何となく書きつけていると、妙に気持ちが高ぶってくる。
行く河の流れは絶えずして、しかももとの水にあらず。
   訳 (指示のみ): 「流れる川の姿は常に変わるが、その水自体は元のものではない。」
   訳 (例を示す): 「流れる川水は絶えることなく、しかし元の水と同じではない。」
   参考: 流れていく川の流れは絶えることがなく、それでいて、もとの水ではない。
男もすなる日記といふものを、女もしてみむとてするなり。
   訳 (指示のみ): 「

In [3]:
import sacrebleu

for label, prompt in [("指示のみ", PLAIN), ("例を示す", FEWSHOT)]:
    hyps = [chat(prompt.format(src)) for src, _ in data]
    print(label, "chrF =", round(sacrebleu.corpus_chrf(hyps, [[ref for _, ref in data]]).score, 1))
print("訳さずにそのまま出した場合 chrF =", round(sacrebleu.corpus_chrf([src for src, _ in data], [[ref for _, ref in data]]).score, 1))

指示のみ chrF = 35.0
例を示す chrF = 36.4
訳さずにそのまま出した場合 chrF = 17.7


In [4]:
# 誤訳しやすい語（古今異義語）の語釈を与えてから訳させる
GLOSSARY = {"あやし": "不思議だ、妙だ (現代語の「怪しい」とは違う)", "ものぐるほし": "気持ちが高ぶる、正気を失ったようだ", "つれづれなり": "することがなく退屈だ",
            "よしなしごと": "とりとめのないこと", "すなる": "する + 伝聞の「なり」= するという", "やうやう": "だんだん、しだいに"}
src = data[3][0]
notes = "\n".join(f"- {k}: {v}" for k, v in GLOSSARY.items() if k[:3] in src)
print(notes)
print(chat(f"次の古文を現代語に訳してください。語釈を参考にし、訳文だけを 1 行で出力してください。\n\n語釈:\n{notes}\n\n古文: {src}"))

- あやし: 不思議だ、妙だ (現代語の「怪しい」とは違う)
- ものぐるほし: 気持ちが高ぶる、正気を失ったようだ
- つれづれなり: することがなく退屈だ
- よしなしごと: とりとめのないこと
手持ち無沙汰なので一日中、硯に向かってとりとめのない thoughts を漫然と書きつけていると、不思議なほどに気持ちが高ぶってくる。


- 古文から現代語への翻訳は、同じ言語の中での「時代をまたぐ翻訳」。語順はほぼ同じなので、難しさは語彙（古今異義語）、助動詞の意味、省略された主語や敬語の解釈にある。
- LLM は有名な作品の冒頭なら訳を覚えていることが多い。知られていない文での性能は、それより低いと考える。評価には、モデルが見たことのなさそうな文を使う。
- この実験では、例を示しても訳はほとんど変わらなかった（chrF は「指示のみ」が 35.0、「例を示す」が 36.4。T4 と同じ float16 で実行した値で、CPU (float32) では 35.9 と 36.0 だった。6 文しかないので、この程度の差は誤差や実行環境による揺れの範囲）。どちらの訳も「ものぐるほし」を「熱中してしまう」と訳したが、語釈を与えると正しく「気持ちが高ぶってくる」と訳された。ただしそのときの訳文には、英単語 (thoughts) が混ざる失敗も出ている。
- 小さなモデルは、古語を現代語の似た語と取り違えることがある（「あやし」を「怪しい」と訳す、など）。語釈や品詞分解の結果（日本語の形態素解析と文分割の章の古文のレシピ）をプロンプトに入れると、誤訳を減らせることがある。
- 対訳データがあれば、この章のファインチューニングのレシピがそのまま使える。現代語の単語ベクトルを古文に適応させる、古文の単言語データで逆翻訳する、といった工夫が研究されている。
- 参考訳は 1 つの解釈にすぎず、直訳か意訳かでも大きく変わる。自動評価の数値は目安にとどめる。